# Notebook 8: `bangla-speech-processing/BanglaASR`

Family: `whisper_hf` · Python 3.12 (Kaggle default) · GPU recommended.

Produces `predictions/bangla-speech-processing__BanglaASR.csv`. Download at the end; upload to notebook 10 for judging.


In [ ]:
# === Clone the benchmark repo ===========================================
# Kaggle paths: /kaggle/working is the writable home. We clone the project
# repo there and cd into it so the relative paths used by src/ resolve.

!apt-get -qq install -y libsndfile1

GITHUB_REPO_URL = "https://github.com/Mainul/banglish-stt-benchmark.git"
REPO_DIR = "/kaggle/working/banglish-stt-benchmark"

import os, sys, subprocess
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", GITHUB_REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print("cwd:", os.getcwd())


In [ ]:
# === Install transformers + base deps ===================================
# These versions are the same across all whisper_hf notebooks (models
# 1, 2, 3, 8). Pinned to match what the project's adapters expect.

!pip install -q transformers==4.44.2
!pip install -q accelerate==0.34.2
!pip install -q librosa==0.10.2.post1
!pip install -q soundfile==0.12.1
!pip install -q pandas==2.2.3
!pip install -q pyyaml==6.0.2
!pip install -q tqdm==4.67.1
!pip install -q requests==2.32.3


In [ ]:
# === Build the 50-clip manifest =========================================
# Downloads OpenSLR-104 if not already cached, extracts, parses transcripts,
# computes length + code-switch density, and stratified-samples 50 clips.
#
# CRITICAL: every notebook in this benchmark calls this with seed=42 so
# they all see the EXACT SAME 50 clips. Do not change the seed.

from src.utils import setup_logging, set_seeds
from src.data_prep import build_manifest

setup_logging("logs")
set_seeds(42)

manifest_path = build_manifest(
    raw_dir="data/raw",
    manifest_path="data/manifest.csv",
    n_samples=50,
    seed=42,
)
print("manifest:", manifest_path)


In [ ]:
# === Transcribe with this model =========================================
# Reads the spec for THIS notebook's model from config/models.yaml,
# then runs the 50-clip loop. Resumable: rerun this cell to pick up
# where it stopped (clips already in the CSV are skipped).

from src.utils import read_yaml
from src.transcribe import run_single_model

MODEL_ID = "bangla-speech-processing/BanglaASR"
spec = next(s for s in read_yaml("config/models.yaml")["models"] if s["id"] == MODEL_ID)

failure = run_single_model(
    spec,
    manifest_csv="data/manifest.csv",
    predictions_dir="predictions",
    device="cuda",
)
print("FAILED:", failure) if failure else print("OK — see predictions/")


In [ ]:
# === Download the predictions CSV =======================================
# Right-click the link below and save to your local machine. You will
# upload all 9 CSVs to the judge notebook (notebook 10) at the end.
#
# File:   predictions/bangla-speech-processing__BanglaASR.csv
# Schema: clip_id, model_id, predicted_text, latency_sec, decoder_variant

from pathlib import Path
import pandas as pd
from IPython.display import FileLink, display

csv_path = Path("predictions/bangla-speech-processing__BanglaASR.csv")
if csv_path.exists():
    df = pd.read_csv(csv_path)
    print(f"{len(df)} rows in {csv_path}")
    display(df.head())
    display(FileLink(str(csv_path)))
else:
    print(f"NOT FOUND: {csv_path} — transcription cell failed or was skipped.")
